In [1]:
from vllm import LLM, SamplingParams
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import re
model_base = "/project/pi_hongyu_umass_edu/shared_llm_checkpoints/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B"
snapshots_dir = os.path.join(model_base, "snapshots")
snapshot = os.listdir(snapshots_dir)[0]  # Get the first (and probably only) snapshot
model_path = os.path.join(snapshots_dir, snapshot)

In [2]:
import torch
def monitor_gpu_memory(message=""):
    """Monitor GPU memory usage"""
    allocated = torch.cuda.memory_allocated() / 1024**2  # Convert to MB
    reserved = torch.cuda.memory_reserved() / 1024**2
    print(f"\nGPU Memory ({message}):")
    print(f"Allocated: {allocated:.2f} MB")
    print(f"Reserved:  {reserved:.2f} MB")

In [3]:
total_memory = torch.cuda.get_device_properties(0).total_memory
allocated_memory = torch.cuda.memory_allocated()
free_memory = total_memory - allocated_memory
print(f"Total GPU Memory: {total_memory / 1024**2:.2f} MB")
print(f"Allocated Memory: {allocated_memory / 1024**2:.2f} MB")
print(f"Free Memory: {free_memory / 1024**2:.2f} MB")  

Total GPU Memory: 48586.38 MB
Allocated Memory: 0.00 MB
Free Memory: 48586.38 MB


In [ ]:
print(f"Loading from: {model_path}")
monitor_gpu_memory("Before initialization")
# Initialize vLLM engine
# Calculate available GPU memory
total_memory = torch.cuda.get_device_properties(0).total_memory
allocated_memory = torch.cuda.memory_allocated()
free_memory = total_memory - allocated_memory

llm = LLM(
    model=model_path,
    trust_remote_code=True,
    dtype="float16",
    gpu_memory_utilization=0.6,  # Reduced from 0.8
    tensor_parallel_size=1,
    max_num_batched_tokens=1024,  # Reduced from 2048
    max_num_seqs=32,  # Reduced from 64
    max_model_len=512,  # Reduced from 1024
)

monitor_gpu_memory("After initialization")

Loading from: /project/pi_hongyu_umass_edu/shared_llm_checkpoints/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/530ca3e1ad39d440e182c2e4317aa40f012512fa

GPU Memory (Before initialization):
Allocated: 0.00 MB
Reserved:  0.00 MB
INFO 03-22 22:18:11 __init__.py:207] Automatically detected platform cuda.
WARNING 03-22 22:18:11 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 03-22 22:19:00 config.py:549] This model supports multiple tasks: {'score', 'classify', 'reward', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 03-22 22:19:00 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='/project/pi_hongyu_umass_edu/shared_llm_checkpoints/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/530ca3e1ad39d440e182c2e4317aa40f012512fa', speculative_config=None, tokenizer='/project/pi_hongyu_umass_edu/shared_llm_checkpoints/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/530ca3e1ad39d440e182c2e4317aa40f012512fa', skip_t

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-22 22:22:56 model_runner.py:1115] Loading model weights took 3.3460 GB
INFO 03-22 22:23:00 worker.py:267] Memory profiling takes 3.41 seconds
INFO 03-22 22:23:00 worker.py:267] the current vLLM instance can use total_gpu_memory (47.45GiB) x gpu_memory_utilization (0.60) = 28.47GiB
INFO 03-22 22:23:00 worker.py:267] model weights take 3.35GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 0.24GiB; the rest of the memory reserved for KV Cache is 24.82GiB.
INFO 03-22 22:23:01 executor_base.py:111] # cuda blocks: 58100, # CPU blocks: 9362
INFO 03-22 22:23:01 executor_base.py:116] Maximum concurrency for 512 tokens per request: 1815.62x
